# B2-024-gpu-scientific-ml-capstone — Practice p15 — Solution

**Type:** proof · **Difficulty:** core · **Concepts:** open-ended-model-evaluation

*45 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Solution

**(a) Mean and variance of a sample mean.**
By **linearity of expectation**, $E[\bar X]=E\big[\tfrac1n\sum_iX_i\big]=\tfrac1n\sum_iE[X_i]=\tfrac1n\cdot n\mu=\mu$.
For the variance, first use $\operatorname{Var}(aY)=a^2\operatorname{Var}(Y)$ with $a=1/n$ and $Y=\sum_iX_i$: $\operatorname{Var}(\bar X)=\tfrac1{n^2}\operatorname{Var}\big(\sum_iX_i\big)$.
By the **variance of a sum**, $\operatorname{Var}\big(\sum_iX_i\big)=\sum_i\operatorname{Var}(X_i)+2\sum_{i<j}\operatorname{Cov}(X_i,X_j)$.
By **independence**, $\operatorname{Cov}(X_i,X_j)=0$ for $i\neq j$, so $\operatorname{Var}\big(\sum_iX_i\big)=n\sigma^2$ and $\operatorname{Var}(\bar X)=n\sigma^2/n^2=\sigma^2/n$.

**(b) Variance of the mean paired difference.**
For one seed, the variance of a sum (with coefficients $+1,-1$, using $\operatorname{Var}(-B_s)=\operatorname{Var}(B_s)$ and $\operatorname{Cov}(A_s,-B_s)=-\operatorname{Cov}(A_s,B_s)$) gives
$$\operatorname{Var}(D_s)=\operatorname{Var}(A_s)+\operatorname{Var}(B_s)-2\operatorname{Cov}(A_s,B_s)=\sigma^2+\sigma^2-2\rho\sigma^2=2\sigma^2(1-\rho).$$
The pairs $(A_s,B_s)$ are independent across $s$, so the $D_s$ (each a function of one pair) are independent, with common mean $E[A_s]-E[B_s]$ (linearity) and common variance $2\sigma^2(1-\rho)$.
Applying (a) to $D_1,\dots,D_r$: $\operatorname{Var}(\bar D)=\dfrac{2\sigma^2(1-\rho)}{r}$.

**(c) Paired versus unpaired.**
Unpaired runs use different seeds, so $\rho=0$ and (b) gives $\operatorname{Var}_{\text{unpaired}}=2\sigma^2/r$.
For $\sigma>0$: $\operatorname{Var}_{\text{paired}}<\operatorname{Var}_{\text{unpaired}}\iff 2\sigma^2(1-\rho)/r<2\sigma^2/r\iff 1-\rho<1\iff\rho>0$ (dividing by the positive number $2\sigma^2/r$ preserves the inequality). For $\rho=0$ they are equal and for $\rho<0$ pairing is worse.
The ratio is $\operatorname{Var}_{\text{paired}}/\operatorname{Var}_{\text{unpaired}}=1-\rho$, which is $0.25$ for $\rho=0.75$: a four-fold variance reduction (half the standard deviation).

**(d) Numbers.**
$\operatorname{Var}(\bar D)=\dfrac{2\cdot(0.002)^2\cdot(1-0.75)}{3}=\dfrac{2\cdot4\times10^{-6}\cdot0.25}{3}=\dfrac{2\times10^{-6}}{3}\approx6.667\times10^{-7}$, so $\operatorname{sd}(\bar D)=\sqrt{6.667\times10^{-7}}\approx8.165\times10^{-4}$.
Two standard deviations are $\approx1.633\times10^{-3}$. The observed $\bar D=0.0015$ is about $1.84$ standard deviations, so it is **not** larger than two standard deviations: the difference is not demonstrated at this level.

**(e) Reusing the validation examples.**
In (a) the averaged quantities are independent; when every seed's score is computed on the same 200 validation examples, the scores of different seeds (and of both configurations) share the same example-sampling component, so they are not independent draws of "performance on fresh data".
That does not invalidate (b), because (b) is a statement about variation over seeds with the validation set held fixed: given the fixed examples, the seeds' training randomness is still independent across $s$, and the shared example component is common to $A_s$ and $B_s$, so it is part of the positive correlation $\rho$ and largely cancels in $D_s$.
What (b) does not cover is the uncertainty from which 200 examples were drawn; that is what the paired bootstrap over examples measures.

## Numeric sanity check

In [ ]:
import math
import torch

SEED = 20261022
sigma, r, rho = 0.002, 3, 0.75
var_paired = 2 * sigma ** 2 * (1 - rho) / r
var_unpaired = 2 * sigma ** 2 / r
sd_paired = math.sqrt(var_paired)
z_observed = 0.0015 / sd_paired

# Monte Carlo: correlated pairs (A_s, B_s) with Var = sigma^2 and Cov = rho sigma^2, same mean
gen = torch.Generator().manual_seed(SEED)
n_rep = 200_000
z1 = torch.randn(n_rep, r, generator=gen, dtype=torch.float64)
z2 = torch.randn(n_rep, r, generator=gen, dtype=torch.float64)
A = sigma * z1
B = sigma * (rho * z1 + math.sqrt(1 - rho ** 2) * z2)
D_bar = (A - B).mean(dim=1)
mc_var = D_bar.var().item()
print(f"Var(D_bar) = {var_paired:.4e} (Monte Carlo {mc_var:.4e}); sd = {sd_paired:.4e}; ratio = {var_paired / var_unpaired}")
print(f"2 sd = {2 * sd_paired:.4e}; observed 0.0015 is {z_observed:.3f} sd")

### Answer check

In [ ]:
assert abs(var_paired - 2e-6 / 3) < 1e-18
assert abs(var_paired / var_unpaired - 0.25) < 1e-12
assert abs(sd_paired - 8.16496580927726e-4) < 1e-12
assert 0.0015 < 2 * sd_paired and abs(z_observed - 1.837) < 1e-3
assert abs(mc_var / var_paired - 1) < 0.02